# Calcula métricas importantes de como o gerador deve funcionar. Isso permite que ele gere pipelines com maior score.

In [14]:
import pandas as pd
import numpy as np
import json
import os
from collections import Counter

In [3]:
"""
Extrai as colunas categóricas de um meta-df para 
obter informações relevantes que serão usadas no 
script de geração de pipelines
"""
def calc_metadf_metrics(dataset):
    data_path = f"../data/meta/meta_processed/meta_proc_{dataset}.csv"
    df = pd.read_csv(data_path)
    print(f"extraindo métricas do dataframe: {dataset.upper()}\n")

    # Filtra apenas as colunas caetgóricas (FLAGS de algoritmos)
    flags = [c for c in df.columns[2:-3] if '-' not in c]
    print(f"quantidade de colunas categóricas: {len(flags)}")

    # extrai o grupo 
    groups = [f.split('.')[0] for f in flags]

    # grupos únicos
    unique_groups = sorted(list(set(groups)))

    # exemplos por grupo
    count = Counter(groups)

    print("Número de grupos:", len(unique_groups))
    print("grupos:", unique_groups)
    print("Quantidade de exemplos por grupo:", count)
    print("\n---------------------------------------------------------------\n")
    # Identifica o top 10%
    top_10_threshold = df['F1 (macro averaged by label)'].quantile(0.90)
    df_elite = df[df['F1 (macro averaged by label)'] >= top_10_threshold].copy()

    # Prints
    print("=== Estatísticas do top 10% pipelines: ===")
    print(f"Threshold F1 para o top 10%: {top_10_threshold:.4f}")
    print(f"Tamanho do top 10%: {len(df_elite)} instâncias\n")

    # Calcula a frequência de normalize e 
    # feature preprocessing no top 10%
    normalize_counts = df['normalize'].mean()
    feature_prep_counts = df_elite['feature preprocessing'].mean()
    print(f"Frequência de normalize = {normalize_counts:.4f}")
    print(f"Frequência de feature preprocessing = {feature_prep_counts:.4f}")
    print("\n---------------------------------------------------------------\n")

    # Loop por cada grupo
    print("Estatísticas por grupo:")
    for g in unique_groups:
        # Identifica as colunas que pertencem a este grupo específico
        group_cols = [f for f in flags if f.split('.')[0] == g]
        
        # Conta quantos algoritmos o grupo possui simultaneamente
        active_counts = df_elite[group_cols].sum(axis=1)
        
        print(f"\n[ Grupo: {g.upper()} ]")
        
        # Itera até o maior número de algoritmos encontrados
        for n in range(int(active_counts.max()) + 1):
            # Calcula a frequência dos algoritmos sozinhos ou em emsemble
            percentage = (active_counts == n).mean() 
            
            if percentage > 0:
                print(f"  - {percentage:.2f} pipelines usam exatamente {n} algoritmos")

    print("\n---------------------------------------------------------------\n")
    print("Top 5 melhores pipelines")
    top5 = df_elite.sort_values(by='F1 (macro averaged by label)', ascending=False).head(5)

    for i, (idx, row) in enumerate(top5.iterrows(), 1):
        pipeline = [f for f in flags if row[f] == 1]
        f1 = row['F1 (macro averaged by label)']
        print(f"{i}º Lugar (F1: {f1:.4f}) -> {pipeline}")
    return


In [28]:
calc_metadf_metrics(dataset="birds")

extraindo métricas do dataframe: BIRDS

quantidade de colunas categóricas: 60
Número de grupos: 4
grupos: ['meka', 'mlfs', 'sklearn', 'weka']
Quantidade de exemplos por grupo: Counter({'weka': 22, 'meka': 21, 'mlfs': 15, 'sklearn': 2})

---------------------------------------------------------------

=== Estatísticas do top 10% pipelines: ===
Threshold F1 para o top 10%: 0.4950
Tamanho do top 10%: 10554 instâncias

Frequência de normalize = 0.1019
Frequência de feature preprocessing = 0.9818

---------------------------------------------------------------

Estatísticas por grupo:

[ Grupo: MEKA ]
  - 1.00% pipelines usam exatamente 1 algoritmos

[ Grupo: MLFS ]
  - 0.26% pipelines usam exatamente 0 algoritmos
  - 0.74% pipelines usam exatamente 1 algoritmos

[ Grupo: SKLEARN ]
  - 0.76% pipelines usam exatamente 0 algoritmos
  - 0.24% pipelines usam exatamente 1 algoritmos

[ Grupo: WEKA ]
  - 0.00% pipelines usam exatamente 0 algoritmos
  - 0.99% pipelines usam exatamente 1 algoritmos

In [29]:
calc_metadf_metrics(dataset="medical")

extraindo métricas do dataframe: MEDICAL

quantidade de colunas categóricas: 60
Número de grupos: 4
grupos: ['meka', 'mlfs', 'sklearn', 'weka']
Quantidade de exemplos por grupo: Counter({'weka': 22, 'meka': 21, 'mlfs': 15, 'sklearn': 2})

---------------------------------------------------------------

=== Estatísticas do top 10% pipelines: ===
Threshold F1 para o top 10%: 0.7380
Tamanho do top 10%: 7500 instâncias

Frequência de normalize = 0.0000
Frequência de feature preprocessing = 0.8923

---------------------------------------------------------------

Estatísticas por grupo:

[ Grupo: MEKA ]
  - 1.00% pipelines usam exatamente 1 algoritmos

[ Grupo: MLFS ]
  - 0.22% pipelines usam exatamente 0 algoritmos
  - 0.78% pipelines usam exatamente 1 algoritmos

[ Grupo: SKLEARN ]
  - 0.89% pipelines usam exatamente 0 algoritmos
  - 0.11% pipelines usam exatamente 1 algoritmos

[ Grupo: WEKA ]
  - 0.12% pipelines usam exatamente 0 algoritmos
  - 0.88% pipelines usam exatamente 1 algoritmo

In [30]:
calc_metadf_metrics(dataset="enron")

extraindo métricas do dataframe: ENRON

quantidade de colunas categóricas: 60
Número de grupos: 4
grupos: ['meka', 'mlfs', 'sklearn', 'weka']
Quantidade de exemplos por grupo: Counter({'weka': 22, 'meka': 21, 'mlfs': 15, 'sklearn': 2})

---------------------------------------------------------------

=== Estatísticas do top 10% pipelines: ===
Threshold F1 para o top 10%: 0.2230
Tamanho do top 10%: 8289 instâncias

Frequência de normalize = 0.0000
Frequência de feature preprocessing = 0.9257

---------------------------------------------------------------

Estatísticas por grupo:

[ Grupo: MEKA ]
  - 1.00% pipelines usam exatamente 1 algoritmos

[ Grupo: MLFS ]
  - 0.41% pipelines usam exatamente 0 algoritmos
  - 0.59% pipelines usam exatamente 1 algoritmos

[ Grupo: SKLEARN ]
  - 0.67% pipelines usam exatamente 0 algoritmos
  - 0.33% pipelines usam exatamente 1 algoritmos

[ Grupo: WEKA ]
  - 0.20% pipelines usam exatamente 0 algoritmos
  - 0.76% pipelines usam exatamente 1 algoritmos


In [31]:
calc_metadf_metrics(dataset="scene")

extraindo métricas do dataframe: SCENE

quantidade de colunas categóricas: 60
Número de grupos: 4
grupos: ['meka', 'mlfs', 'sklearn', 'weka']
Quantidade de exemplos por grupo: Counter({'weka': 22, 'meka': 21, 'mlfs': 15, 'sklearn': 2})

---------------------------------------------------------------

=== Estatísticas do top 10% pipelines: ===
Threshold F1 para o top 10%: 0.7480
Tamanho do top 10%: 7924 instâncias

Frequência de normalize = 0.0847
Frequência de feature preprocessing = 0.8087

---------------------------------------------------------------

Estatísticas por grupo:

[ Grupo: MEKA ]
  - 1.00% pipelines usam exatamente 1 algoritmos

[ Grupo: MLFS ]
  - 0.34% pipelines usam exatamente 0 algoritmos
  - 0.66% pipelines usam exatamente 1 algoritmos

[ Grupo: SKLEARN ]
  - 0.85% pipelines usam exatamente 0 algoritmos
  - 0.15% pipelines usam exatamente 1 algoritmos

[ Grupo: WEKA ]
  - 0.74% pipelines usam exatamente 1 algoritmos
  - 0.26% pipelines usam exatamente 2 algoritmos


In [32]:
calc_metadf_metrics(dataset="yeast")

extraindo métricas do dataframe: YEAST

quantidade de colunas categóricas: 60
Número de grupos: 4
grupos: ['meka', 'mlfs', 'sklearn', 'weka']
Quantidade de exemplos por grupo: Counter({'weka': 22, 'meka': 21, 'mlfs': 15, 'sklearn': 2})

---------------------------------------------------------------

=== Estatísticas do top 10% pipelines: ===
Threshold F1 para o top 10%: 0.4360
Tamanho do top 10%: 9537 instâncias

Frequência de normalize = 0.0890
Frequência de feature preprocessing = 0.9521

---------------------------------------------------------------

Estatísticas por grupo:

[ Grupo: MEKA ]
  - 1.00% pipelines usam exatamente 1 algoritmos

[ Grupo: MLFS ]
  - 0.17% pipelines usam exatamente 0 algoritmos
  - 0.83% pipelines usam exatamente 1 algoritmos

[ Grupo: SKLEARN ]
  - 0.88% pipelines usam exatamente 0 algoritmos
  - 0.12% pipelines usam exatamente 1 algoritmos

[ Grupo: WEKA ]
  - 0.70% pipelines usam exatamente 1 algoritmos
  - 0.30% pipelines usam exatamente 2 algoritmos


## Acima estão resumidos os padrões identificados no top 10% de cada meta-conjunto. O objetivo é descobrir algumas configurações de pipeline para orientar a geração de novos candidatos. Os padrões usados estão disponíveis no arquivo JSON.

## Agora precisamos descobrir qual intervalo os hiperparâmetros estão concentrados. Isso permitirá com que o modelo faça a inferência com mais precisão

In [7]:
def get_hiperparameter_range(dataset: str):

    data_path = f"../data/meta/meta_processed/meta_proc_{dataset}.csv"
    config_dir = "../configs"
    output_file = f"{config_dir}/hyperparameter_ranges_{dataset}.json"

    # Carrega os dados
    df = pd.read_csv(data_path)

    # As colunas que mão são flags (categóricas) são os hiperparâmetros
    all_relevant_cols = df.columns[2:-3]
    flags = [c for c in all_relevant_cols if '-' not in c]
    hyperparams = [c for c in all_relevant_cols if c not in flags]

    print(f"Extraindo limites para {len(hyperparams)} hiperparâmetros...")

    # 2. Dicionário para armazenar os limites
    hp_ranges = {}

    for col in hyperparams:
        # Filtra apenas os valores válidos (maiores que -1)
        valid_values = df[col][df[col] > -1]
        
        if not valid_values.empty:
            hp_ranges[col] = {
                "min": float(valid_values.min()),
                "max": float(valid_values.max()),
                "mean": float(valid_values.mean()) # Útil caso queira usar a média como fallback
            }

    # 3. Salva o resultado em um JSON
    with open(output_file, 'w') as f:
        json.dump(hp_ranges, f, indent=4)

    print(f"Sucesso! As fronteiras foram salvas em: {output_file}")

    # Exemplo de visualização dos primeiros 10 parâmetros extraídos
    print("\nExemplo de limites extraídos:")
    for k in list(hp_ranges.keys())[:10]:
        print(f"  {k}: {hp_ranges[k]['min']} to {hp_ranges[k]['max']}")

In [8]:
get_hiperparameter_range("birds")

Extraindo limites para 145 hiperparâmetros...
Sucesso! As fronteiras foram salvas em: ../configs/hyperparameter_ranges_birds.json

Exemplo de limites extraídos:
  mlfs.igmf_adapted.PyIT_IGMF-n_features: 0.0 to 78.0
  mlfs.ppt_mi_adapted.PyIT_PPT_MI-n_features: 0.0 to 78.0
  mlfs.scls_adapted.PyIT_SCLS-n_features: 0.0 to 78.0
  mlfs.lrfs_adapted.PyIT_LRFS-n_features: 2.0 to 78.0
  mlfs.mlsmfs_adapted.PyIT_MLSMFS-n_features: 0.0 to 77.0
  sklearn.feature_selection.SelectFromModel-estimator: 0.0 to 1.0
  sklearn.feature_selection.SelectFromModel-threshold: 0.0 to 0.0
  sklearn.feature_selection.RFE-estimator: 0.0 to 1.0
  sklearn.feature_selection.RFE-n_features_to_select: 0.0 to 78.0
  mlfs.br_skb.BR_SelectKBest-n_features: 0.0 to 78.0


In [9]:
get_hiperparameter_range("medical")

Extraindo limites para 145 hiperparâmetros...
Sucesso! As fronteiras foram salvas em: ../configs/hyperparameter_ranges_medical.json

Exemplo de limites extraídos:
  mlfs.igmf_adapted.PyIT_IGMF-n_features: 0.0 to 145.0
  mlfs.ppt_mi_adapted.PyIT_PPT_MI-n_features: 0.0 to 145.0
  sklearn.feature_selection.SelectFromModel-estimator: 0.0 to 1.0
  sklearn.feature_selection.SelectFromModel-threshold: 0.0 to 0.0
  mlfs.br_skb.BR_SelectKBest-n_features: 0.0 to 145.0
  mlfs.br_skb.BR_SelectKBest-method: 0.0 to 2.0
  mlfs.ppt_skb.PPT_SelectKBest-n_features: 0.0 to 145.0
  mlfs.ppt_skb.PPT_SelectKBest-method: 0.0 to 2.0
  mlfs.br_relieff.BR_ReliefF-n_features: 0.0 to 145.0
  mlfs.br_relieff.BR_ReliefF-neighbors: 0.0 to 0.0


In [10]:
get_hiperparameter_range("enron")

Extraindo limites para 145 hiperparâmetros...
Sucesso! As fronteiras foram salvas em: ../configs/hyperparameter_ranges_enron.json

Exemplo de limites extraídos:
  mlfs.igmf_adapted.PyIT_IGMF-n_features: 2.0 to 99.0
  mlfs.ppt_mi_adapted.PyIT_PPT_MI-n_features: 0.0 to 100.0
  sklearn.feature_selection.SelectFromModel-estimator: 0.0 to 1.0
  sklearn.feature_selection.SelectFromModel-threshold: 0.0 to 0.0
  mlfs.br_skb.BR_SelectKBest-n_features: 0.0 to 100.0
  mlfs.br_skb.BR_SelectKBest-method: 0.0 to 2.0
  mlfs.ppt_skb.PPT_SelectKBest-n_features: 0.0 to 100.0
  mlfs.ppt_skb.PPT_SelectKBest-method: 0.0 to 2.0
  mlfs.ppt_relieff.PPT_ReliefF-n_features: 0.0 to 100.0
  mlfs.ppt_relieff.PPT_ReliefF-neighbors: 0.0 to 0.0


In [11]:
get_hiperparameter_range("scene")

Extraindo limites para 145 hiperparâmetros...
Sucesso! As fronteiras foram salvas em: ../configs/hyperparameter_ranges_scene.json

Exemplo de limites extraídos:
  mlfs.igmf_adapted.PyIT_IGMF-n_features: 0.0 to 88.0
  mlfs.ppt_mi_adapted.PyIT_PPT_MI-n_features: 0.0 to 88.0
  mlfs.scls_adapted.PyIT_SCLS-n_features: 0.0 to 88.0
  mlfs.lrfs_adapted.PyIT_LRFS-n_features: 0.0 to 86.0
  mlfs.lsmfs_adapted.PyIT_LSMFS-n_features: 0.0 to 88.0
  mlfs.mlsmfs_adapted.PyIT_MLSMFS-n_features: 0.0 to 88.0
  sklearn.feature_selection.SelectFromModel-estimator: 0.0 to 1.0
  sklearn.feature_selection.SelectFromModel-threshold: 0.0 to 0.0
  sklearn.feature_selection.RFE-estimator: 0.0 to 0.0
  sklearn.feature_selection.RFE-n_features_to_select: 0.0 to 88.0


In [12]:
get_hiperparameter_range("yeast")

Extraindo limites para 145 hiperparâmetros...
Sucesso! As fronteiras foram salvas em: ../configs/hyperparameter_ranges_yeast.json

Exemplo de limites extraídos:
  mlfs.igmf_adapted.PyIT_IGMF-n_features: 0.0 to 31.0
  mlfs.ppt_mi_adapted.PyIT_PPT_MI-n_features: 0.0 to 31.0
  mlfs.scls_adapted.PyIT_SCLS-n_features: 0.0 to 31.0
  mlfs.lrfs_adapted.PyIT_LRFS-n_features: 0.0 to 31.0
  mlfs.lsmfs_adapted.PyIT_LSMFS-n_features: 0.0 to 31.0
  mlfs.mlsmfs_adapted.PyIT_MLSMFS-n_features: 0.0 to 31.0
  sklearn.feature_selection.SelectFromModel-estimator: 0.0 to 1.0
  sklearn.feature_selection.SelectFromModel-threshold: 0.0 to 0.0
  sklearn.feature_selection.RFE-estimator: 0.0 to 1.0
  sklearn.feature_selection.RFE-n_features_to_select: 0.0 to 31.0


## Após extrair os intervalos individuais de cada meta-conjunto, vamos unir estes intervalos em um único JSON. Isso simplifica a lógica de busca e permite que o modelo generalize mais, podendo encontrar configurações melhores

In [ ]:
CONFIG_DIR = "../configs"
OUTPUT_FILE = f"{CONFIG_DIR}/global_hyperparameter_ranges.json"
MAX_EXPANSION_THRESHOLD = 2.0  # Parâmetros que foram alterados em mais de 2x

def merge_hyperparameters_audit(threshold):
    # Carrega todos os arquivos JSON de ranges individuais
    json_files = [f for f in os.listdir(CONFIG_DIR) if f.startswith("hyperparameter_ranges_")]
    
    all_data = []
    for f in json_files:
        with open(os.path.join(CONFIG_DIR, f), 'r') as file:
            all_data.append(json.load(file))
            
    if not all_data:
        print("Erro: Nenhum arquivo de range individual encontrado em '../configs'.")
        return

    all_params = set().union(*(d.keys() for d in all_data))
    global_ranges = {}
    alert_count = 0

    print(f"AUDITORIA DE RANGES GLOBAIS (Threshold: {threshold}x)")
    print(f"Analisando {len(all_params)} parâmetros entre {len(json_files)} datasets...\n")

    for param in sorted(all_params):
        # Coleta min, max e ranges locais
        mins = [d[param]['min'] for d in all_data if param in d]
        maxs = [d[param]['max'] for d in all_data if param in d]
        local_ranges = [d[param]['max'] - d[param]['min'] for d in all_data if param in d]
        
        # Valores globais absolutos
        abs_min = min(mins)
        abs_max = max(maxs)
        global_range = abs_max - abs_min
        avg_local_range = np.mean(local_ranges)
        
        # Verificação de Sanidade (Alerta se range global alterou > 2X)
        if avg_local_range > 0:
            expansion_factor = global_range / avg_local_range
            
            if expansion_factor > threshold:
                alert_count += 1
                print(f"[ALERT] {param[:50]:<50} | Expansão: {expansion_factor:6.2f}x | Global: [{abs_min:.4f} to {abs_max:.4f}]")
        
        # Salva o valor real
        global_ranges[param] = {
            "min": float(abs_min),
            "max": float(abs_max)
        }

    # Salva o JSON Global Único
    with open(OUTPUT_FILE, 'w') as f:
        json.dump(global_ranges, f, indent=4)
    
    print("\n" + "="*80)
    print(f"RELATÓRIO FINAL:")
    print(f"- Total de parâmetros processados: {len(all_params)}")
    print(f"- Parâmetros com expansão crítica (> {threshold}x): {alert_count}")
    print(f"- Arquivo salvo em: {OUTPUT_FILE}")
    print("="*80)

# Executa a auditoria e fusão
merge_hyperparameters_audit(threshold=MAX_EXPANSION_THRESHOLD)

--- AUDITORIA DE RANGES GLOBAIS (Threshold: 2.0x) ---
Analisando 142 parâmetros entre 5 datasets...

[ALERT] meka.classifiers.multilabel.BPNN-H                 | Expansão:   2.31x | Global: [0.0000 to 1131.0000]
[ALERT] meka.classifiers.multilabel.CDT-L                  | Expansão:   5.00x | Global: [0.0000 to 3.0000]
[ALERT] meka.classifiers.multilabel.RAkEL-M                | Expansão:   2.23x | Global: [0.0000 to 82.0000]
[ALERT] meka.classifiers.multilabel.RAkEL-k                | Expansão:   2.33x | Global: [0.0000 to 20.0000]
[ALERT] meka.classifiers.multilabel.RAkELd-batch-size      | Expansão:   2.23x | Global: [0.0000 to 82.0000]

RELATÓRIO FINAL:
- Total de parâmetros processados: 142
- Parâmetros com expansão crítica (> 2.0x): 5
- Arquivo salvo em: ../configs/global_hyperparameter_ranges.json


## Por úlitmo, vamos salvar o cabeçalho dos meta-conjuntos em um JSON, para podermos carregá-lo no script

In [19]:
dataset_ref = "birds" # Qualquer um serve, pois todos têm o mesmo shape
input_path = f"../data/meta/meta_processed/meta_proc_{dataset_ref}.csv"
output_path = "../configs/feature_schema.json"

# Lê apenas o cabeçalho 
df_headers = pd.read_csv(input_path, nrows=0)

# Extrai exatamente o miolo das features (ignora normalize, feature preprocessing e Targets)
feature_list = df_headers.columns[2:-3].tolist()

# Salva o "contrato" de colunas
with open(output_path, 'w') as f:
    json.dump(feature_list, f, indent=4)

print(f"Schema com {len(feature_list)} colunas salvo em: {output_path}")

Schema com 205 colunas salvo em: ../configs/feature_schema.json


## Com isso, temos o JSON que define a probabilidade do gerador escolher um algoritmo ao invés do outro, e também temos os ranges que os hiperparâmetros daquele algoritmo estão. Assim, o gerador têm tudo para criar pipelines candidatos. O gerador está disponível em `/src`, no arquivo `pipeline_generator.py` 